# 🕷️ Web Scraping in Code (requests + BeautifulSoup)

Welcome to the hands-on companion of the Module 04 reading page. The theory (what scraping is, page anatomy, how the request works) lives on the reading page — here we write every line of code.

This notebook runs **fully offline**: instead of a live site we parse two bundled snapshot pages (`sample_jobs_1.html`, `sample_jobs_2.html`) that mimic a job board. The technique is exactly the same as on a real site — only the selectors differ (Chapter 8 explains why).

---

## 📚 Table of Contents

1. ⚙️ Chapter 1: Setup
2. 📄 Chapter 2: Loading a Page
3. 🔍 Chapter 3: find() vs find_all()
4. 🧺 Chapter 4: The Five Lists (titles, links, occupations, companies, specs)
5. 📦 Chapter 5: Dict → DataFrame → CSV
6. 🔁 Chapter 6: Pagination
7. 🧰 Chapter 7: The Helper Module
8. 🌐 Chapter 8: Going Live (and Why Sites Say No)
9. 📌 Final Summary & Cheat Sheet

# ⚙️ Chapter 1: Setup

### 🎯 Goal:
Prepare the four tools: `requests` (fetch), `bs4` (parse), `lxml` (fast parser), `pandas` (pack results).

## 📦 Install (once)
- *What:* One install per machine.
- *Why:* Colab/Kaggle already ship these; a fresh laptop does not.
- *When to use:* Run once, then forget it.

In [1]:
# Run once if imports below fail:
# !pip install requests bs4 lxml pandas

## 📥 Imports

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import math

# 📄 Chapter 2: Loading a Page

### 🎯 Goal:
Turn raw HTML into a searchable `soup` object — the parsed tree every later step queries.
## 📂 Local sample instead of a live request
- *What:* We open `sample_jobs_1.html`, a 3-job snapshot of a job board saved next to this notebook.
- *Why:* Real job boards change markup and block bots (see Chapter 8) — the snapshot lets every cell below run deterministically, on any machine, forever.
- *When to use:* Practice parsing here; point the same code at a live URL (with permission) when it is your turn.

In [3]:
with open('sample_jobs_1.html', encoding='utf-8') as f:
    html = f.read()

soup = BeautifulSoup(html, 'lxml')
type(soup)

bs4.BeautifulSoup

In [4]:
soup.title.text  # quick proof the tree parsed: the page title

'Sample Jobs - Page 1'

📌 **Insight:** `BeautifulSoup(html, 'lxml')` is always the same two arguments — *page source in, searchable tree out* — whether the source came from a file (here) or `response.content` (live).

# 🔍 Chapter 3: find() vs find_all()

### 🎯 Goal:
Probe **one** element with `find()`, then collect **all** of them with `find_all()`.
## 🎯 First: one title
- *What:* `find()` returns the **first** match only.
- *Why:* Cheap way to inspect an element's shape before collecting the batch.
- *When to use:* Exploring a new page.

In [5]:
title = soup.find('h2', {'class': 'job-title'})
title

<h2 class="job-title"><a class="job-link" href="/jobs/machine-learning-engineer-cairo">Machine Learning Engineer</a></h2>

In [6]:
title.a.text  # the text of its one <a> child

'Machine Learning Engineer'

## 🎯 Then: every title
- *What:* `find_all()` returns a **list** of every match.
- *Why:* One element proves the selector; the list is the dataset.

In [7]:
titles = soup.find_all('h2', {'class': 'job-title'})
len(titles)  # one per job card

3

In [8]:
titles[0].text

'Machine Learning Engineer'

In [9]:
for title in titles:
    print(title.text)

Machine Learning Engineer
Data Scientist
Python Developer


📌 **Insight:** Attribute dictionaries are double-quoted by convention (`{'class': 'job-title'}`) — and `tag.a.text` drills straight to an anchor's text, skipping surrounding whitespace.

# 🧺 Chapter 4: The Five Lists

### 🎯 Goal:
Extract five parallel lists — titles, links, occupations, companies, specs — with one list comprehension each, sanity-checking sizes as you go.

## 📝 1. Titles

In [10]:
titles_lst = [title.a.text for title in titles]
titles_lst

['Machine Learning Engineer', 'Data Scientist', 'Python Developer']

## 🔗 2. Links (relative → absolute)
- *What:* Cards store relative URLs (`/jobs/...`); prepend the origin.
- *Why:* A relative link is useless outside its site — absolute links survive in your CSV.

In [11]:
links = ['https://wuzzuf.net' + title.a['href'] for title in titles]
links

['https://wuzzuf.net/jobs/machine-learning-engineer-cairo',
 'https://wuzzuf.net/jobs/data-scientist-giza',
 'https://wuzzuf.net/jobs/python-developer-alex']

## 💼 3. Occupations (+ the sanity check)
- *What:* Same `find_all` pattern on a different class.
- *Why:* **Counts must agree** — 3 titles but 5 occupations means your selector is catching something extra.

In [12]:
occupations = soup.find_all('div', {'class': 'job-occupation'})
len(occupations), len(titles)  # must match!

(3, 3)

In [13]:
occupations_lst = [occupation.text for occupation in occupations]
occupations_lst

['Engineering - Software', 'Data Science - Analytics', 'Engineering - Backend']

📌 **Insight:** Whenever two lists feed the same table, assert their lengths match before continuing — mismatched columns are the most common silent scraping bug.

## 🏢 4. Companies (+ tiny cleanup)

In [14]:
companies = soup.find_all('a', {'class': 'job-company'})
companies_lst = [company.text for company in companies]
companies_lst

['Edentech -', 'Nile Analytics', 'Delta Soft -']

In [15]:
companies_lst = [company.replace(' -', '') for company in companies_lst]
companies_lst  # trailing ' -' removed

['Edentech', 'Nile Analytics', 'Delta Soft']

## 📋 5. Specs (+ field-count check)
- *What:* Specs pack several facts into one string, separated by `·`.
- *Why:* Verify every row splits into the **same number of fields** before building the table.

In [16]:
specs = soup.find_all('div', {'class': 'job-specs'})
specs[0].text

'Full Time · Cairo · 2 - 4 Years Experience'

In [17]:
specs[0].text.split(' · ')

['Full Time', 'Cairo', '2 - 4 Years Experience']

In [18]:
for spec in specs:
    print(len(spec.text.split(' · ')))  # every row -> same count?

3
3
3


In [19]:
specs_lst = [spec.text for spec in specs]

# 📦 Chapter 5: Dict → DataFrame → CSV

### 🎯 Goal:
Pack five equal-length lists into a table and save it.
## 📦 Why a dict of equal lists?
- *What:* `{'ColumnName': [row, row, ...]}` maps one-to-one onto `pd.DataFrame(...)`.
- *Why:* Five equal lists become a 5-column table — no reshaping needed.

In [20]:
scraped_data = {}
scraped_data['Title'] = titles_lst
scraped_data['Link'] = links
scraped_data['Occupation'] = occupations_lst
scraped_data['Company'] = companies_lst
scraped_data['Specs'] = specs_lst

In [21]:
df = pd.DataFrame(scraped_data)
df

,Title,Link,Occupation,Company,Specs
0,Machine Learning Engineer,https://wuzzuf.net/jobs/machine-learning-engin...,Engineering - Software,Edentech,Full Time · Cairo · 2 - 4 Years Experience
1,Data Scientist,https://wuzzuf.net/jobs/data-scientist-giza,Data Science - Analytics,Nile Analytics,Full Time · Giza · 3 - 5 Years Experience
2,Python Developer,https://wuzzuf.net/jobs/python-developer-alex,Engineering - Backend,Delta Soft,Part Time · Alexandria · 1 - 2 Years Experience


In [22]:
df.to_csv('mljobs.csv', index=False)
print('saved', len(df), 'jobs to mljobs.csv')

saved 3 jobs to mljobs.csv


# 🔁 Chapter 6: Pagination

### 🎯 Goal:
Walk **every** results page, not just the first — read the total, compute pages, loop with `+=`.
## 🔢 1. Reading the total
- *What:* The page prints something like `"1–3 of 6 Jobs"` — split on `'of '` to isolate the number.
- *Why:* You cannot loop pages you cannot count.

In [23]:
s = soup.find('li', {'class': 'job-count'}).text
print(s)

1–3 of 6 Jobs


In [24]:
jobs = int(s.split('of ')[1].split()[0])  # '6 Jobs' -> 6
jobs

6

## 🔢 2. Jobs → pages
- *What:* Divide by jobs-per-page (3 here), round **up** with `math.ceil`.
- *Why:* A partial last page still needs fetching.

In [25]:
PER_PAGE = 3
pages = math.ceil(jobs / PER_PAGE)
pages

2

## 🔢 3. Looping pages with `+=`
- *What:* Re-scrape the same five lists per page, concatenating into the running lists.
- *Why:* One page of results is a toy; the loop is the dataset. (Real sites use `&start=` in the URL; here each "page" is a local snapshot file.)

In [26]:
titles_lst, links_lst = [], []
for pageNo in range(1, pages + 1):
    with open(f'sample_jobs_{pageNo}.html', encoding='utf-8') as f:
        page_soup = BeautifulSoup(f.read(), 'lxml')
    page_titles = page_soup.find_all('h2', {'class': 'job-title'})
    titles_lst += [t.a.text for t in page_titles]
    links_lst += ['https://wuzzuf.net' + t.a['href'] for t in page_titles]
print(len(titles_lst), 'titles across', pages, 'pages')

6 titles across 2 pages


In [27]:
all_df = pd.DataFrame({'Title': titles_lst, 'Link': links_lst})
all_df

,Title,Link
0,Machine Learning Engineer,https://wuzzuf.net/jobs/machine-learning-engin...
1,Data Scientist,https://wuzzuf.net/jobs/data-scientist-giza
2,Python Developer,https://wuzzuf.net/jobs/python-developer-alex
3,MLOps Engineer,https://wuzzuf.net/jobs/ml-ops-engineer-cairo
4,Data Analyst,https://wuzzuf.net/jobs/data-analyst-remote
5,NLP Engineer,https://wuzzuf.net/jobs/nlp-engineer-giza


📌 **Insight:** `+=` on lists is the pagination workhorse — each page appends its batch to the same five running lists, so the final dict-to-DataFrame step (Chapter 5) works unchanged.

# 🧰 Chapter 7: The Helper Module

### 🎯 Goal:
Reuse the real `scrap_helper.py` combiners — the functions that merge multi-query results — on synthetic data, with zero network.
## 🧰 Importing helpers (safe: importing runs no scraping)

In [28]:
from scrap_helper import combine_dfs, combine_dicts

## 🧰 Combining dicts: two queries, one table

In [29]:
q1 = {'Title': ['ML Engineer'], 'Company': ['Edentech']}
q2 = {'Title': ['Data Scientist'], 'Company': ['Nile Analytics']}
combined = combine_dicts([q1, q2])
combined

{'Title': ['ML Engineer', 'Data Scientist'],
 'Company': ['Edentech', 'Nile Analytics']}

## 🧰 Combining DataFrames: concat + dedupe

In [30]:
import pandas as pd
df1 = pd.DataFrame({'Title': ['ML Engineer', 'ML Engineer']})
df2 = pd.DataFrame({'Title': ['Data Scientist']})
combine_dfs([df1, df2])

,Title
0,ML Engineer
0,Data Scientist


📌 **Insight:** `scrap_helper.py` also ships `find_no_of_jobs()` and `scrap_pages()` — they implement Chapters 6 + 4 end-to-end but need a **live** site, so they sit out this offline run. The full single-page script lives in `scrap_jobs.py` (same deal: correct code, needs a reachable site).

# 🌐 Chapter 8: Going Live (and Why Sites Say No)

### 🎯 Goal:
See what happens when this code meets the real Wuzzuf today — and learn the two facts of scraping life: **bot protection** and **selector rot**.
## 🌐 The honest attempt (guarded: it is *expected* to fail)

In [31]:
try:
    r = requests.get('https://wuzzuf.net/search/jobs/?q=machine+learning&a=hpb', timeout=15)
    print('HTTP', r.status_code)
    if r.status_code == 200:
        live = BeautifulSoup(r.content, 'lxml')
        print('titles found:', len(live.find_all('h2', {'class': 'css-m604qf'})))
    else:
        print('Blocked: the site refused a bot-like request (HTTP %d).' % r.status_code)
except Exception as e:
    print(type(e).__name__, '-', e)
    print('No live page: offline snapshots above are the practice ground.')

HTTP 403
Blocked: the site refused a bot-like request (HTTP 403).


📌 **Insight:** Two lessons in one cell — (1) sites *actively* block scrapers (403 here), so always scrape politely and with permission; (2) selectors rot: the lecture's `css-m604qf`-style classes already differ from today's markup. That is why this notebook teaches on readable classes — the *technique* (`find`/`find_all`/lists/DataFrame) never changes, only the selectors do.

---
## 📌 Final Summary

In this notebook you ran the full scraping pipeline end-to-end, offline:

- ⚙️ Setup: `requests` + `bs4` + `lxml` parser + `pandas`
- 📄 Load: file (or `response.content`) → `BeautifulSoup(...)` → searchable tree
- 🔍 Select: `find()` to probe one element, `find_all()` to collect the batch
- 🧺 Extract: five parallel lists, lengths sanity-checked, ` -` cleaned, `·` fields counted
- 📦 Pack: dict of equal lists → `pd.DataFrame` → `mljobs.csv`
- 🔁 Paginate: read the total → `math.ceil(jobs / PER_PAGE)` → loop with `+=`
- 🧰 Reuse: `combine_dfs` / `combine_dicts` from `scrap_helper.py`
- 🌐 Reality: live sites block bots (403) and rotate selectors — technique over selectors

### 🧰 Cheat Sheet

| Task | Code |
|------|------|
| Parse | `BeautifulSoup(html, 'lxml')` |
| Probe / collect | `soup.find("h2", {'class': ...})` / `soup.find_all(...)` |
| Text / link | `tag.a.text`, `tag.a['href']` |
| Clean | `.replace(' -', '')`, `.split(' · ')` |
| Pack | `pd.DataFrame({'Title': titles_lst, ...})`, `df.to_csv(...)` |
| Pages | `math.ceil(jobs / PER_PAGE)`, `for p in range(...)` + `+=` |
| Merge | `combine_dfs([df1, df2])`, `combine_dicts([d1, d2])` |

**Next step:** the reading page’s Coding 03 section shows the finished `scrap_jobs.py` script — one function, five fields, CSV out.